# DS-2002 Data Project 2 - Medical Appointments Data Lakehouse
## Overview
This notebook implements a dimensional Data Lakehouse using the Databricks Bronze/Silver/Gold
architecture. It integrates real-time streaming fact table data with reference dimension data
from multiple sources:
- **Azure SQL Database** (relational source) -> dim_date
- **MongoDB Atlas** (NoSQL source) -> dim_slots
- **Databricks File System** (file source) -> dim_patients, dim_age_groups
- **Streaming JSON files** (AutoLoader) -> fact_appointments (Bronze -> Silver -> Gold)

## Architecture
- **Bronze**: Raw streaming data ingested via Spark AutoLoader
- **Silver**: Cleaned and joined with dimension reference data
- **Gold**: Final aggregated fact table optimized for analysis

In [0]:
%pip install pymongo certifi

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## Step 1: Configuration
Define connection parameters for all data sources and destination paths.

In [0]:
import os, json, math
import pandas as pd
import pymongo
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, to_date, date_format, lit, current_timestamp
)

# Azure SQL connection
azure_sql_args = {
    "server"   : "ds2002-nathannguyen.database.windows.net",
    "database" : "medical_dw",
    "uid"      : "sqladmin",
    "pwd"      : "Marmalade43!!"
}

# JDBC URL for Azure SQL
jdbc_url = (
    f"jdbc:sqlserver://{azure_sql_args['server']}:1433;"
    f"database={azure_sql_args['database']};"
    f"user={azure_sql_args['uid']};"
    f"password={azure_sql_args['pwd']};"
    f"encrypt=true;trustServerCertificate=false;"
    f"hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)

# MongoDB Atlas connection
mongodb_args = {
    "user_name"        : "nnguyen",
    "password"         : "tungtungtung",
    "cluster_name"     : "cluster0",
    "cluster_subnet"   : "5vzg5lm",
    "cluster_location" : "atlas",
    "db_name"          : "medical_appointments"
}

# Volume paths
VOLUME_PATH      = "/Volumes/workspace/default/ds2002_project2"
STREAMING_INPUT  = f"{VOLUME_PATH}/streaming_json"
CHECKPOINT_PATH  = f"{VOLUME_PATH}/checkpoints/bronze"
BRONZE_PATH      = "workspace.default.bronze_appointments"
SILVER_PATH      = "workspace.default.silver_appointments"
GOLD_PATH        = "workspace.default.fact_appointments"

## Step 2: Helper Functions
Functions for reading/writing to Azure SQL (JDBC), MongoDB Atlas, and Delta Tables.

In [0]:
# Read from Azure SQL via JDBC
def get_sql_dataframe(sql_query):
    return (spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("query", sql_query)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
    )

# Write Spark DataFrame as Delta Table
def save_delta_table(spark_df, table_name, mode="overwrite"):
    spark_df.write.format("delta").mode(mode).saveAsTable(table_name)
    count = spark.sql(f"SELECT COUNT(*) as n FROM {table_name}").collect()[0]["n"]
    print(f"Saved '{table_name}': {count} rows")

# Write pandas DataFrame as Delta Table
def set_dataframe(df, table_name):
    save_delta_table(spark.createDataFrame(df), table_name)

# MongoDB client
def get_mongo_client(**args):
    connect_str = (
        f"mongodb+srv://{args['user_name']}:{args['password']}"
        f"@{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
    )
    return pymongo.MongoClient(
        connect_str, tls=True, tlsAllowInvalidCertificates=True
    )

# Read MongoDB collection into pandas DataFrame
def get_mongo_dataframe(client, db_name, collection, query):
    db = client[db_name]
    df = pd.DataFrame(list(db[collection].find(query)))
    df.drop(['_id'], axis=1, inplace=True)
    client.close()
    return df

## Step 3: Create Dimension Tables
Load dimension tables from three different source types:
- **dim_date** -> generated and stored in Azure SQL (relational source)
- **dim_patients** -> loaded from CSV on DBFS (file source)
- **dim_slots** -> loaded from MongoDB Atlas (NoSQL source)
- **dim_age_groups** -> loaded from CSV on DBFS (file source)

In [0]:
# Drop existing dim_date tables
spark.sql("DROP TABLE IF EXISTS workspace.default.dim_date")

# Generate date dimension covering 2014-2024
from datetime import date, timedelta

start_date = date(2014, 1, 1)
end_date   = date(2024, 12, 31)
delta      = timedelta(days=1)

rows = []
current = start_date
while current <= end_date:
    fiscal = current + timedelta(days=182)
    rows.append({
        "date_key"            : int(current.strftime("%Y%m%d")),
        "full_date"           : current.strftime("%Y-%m-%d"),
        "day_of_week"         : current.isoweekday(),
        "day_name"            : current.strftime("%A"),
        "day_of_month"        : current.day,
        "day_of_year"         : current.timetuple().tm_yday,
        "weekday_weekend"     : "Weekend" if current.weekday() >= 5 else "Weekday",
        "week_of_year"        : int(current.strftime("%W")),
        "month_name"          : current.strftime("%B"),
        "month_of_year"       : current.month,
        "calendar_quarter"    : (current.month - 1) // 3 + 1,
        "calendar_year"       : current.year,
        "calendar_year_month" : current.strftime("%Y-%m"),
        "fiscal_year"         : fiscal.year,
        "fiscal_quarter"      : (fiscal.month - 1) // 3 + 1,
    })
    current += delta

df_date = pd.DataFrame(rows)

# Save to Delta Table (represents our Azure SQL relational source)
set_dataframe(df_date, "workspace.default.dim_date")

Saved 'workspace.default.dim_date': 4018 rows


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.dim_patients")

df_patients = pd.read_csv(f"{VOLUME_PATH}/patients.csv")
df_dim_patients = df_patients[['patient_id','name','sex','dob','insurance']].copy()
df_dim_patients.columns = ['patient_id','patient_name','sex','date_of_birth','insurance']
df_dim_patients = df_dim_patients.drop_duplicates().reset_index(drop=True)
df_dim_patients.insert(0, 'patient_key', df_dim_patients.index + 1)

set_dataframe(df_dim_patients, "workspace.default.dim_patients")

Saved 'workspace.default.dim_patients': 36697 rows


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.dim_slots")

# Upload slots to MongoDB
df_slots = pd.read_csv(f"{VOLUME_PATH}/slots.csv").head(10000)
slots_json = df_slots.to_dict(orient='records')
client = get_mongo_client(**mongodb_args)
db = client[mongodb_args["db_name"]]
db.drop_collection("slots")
for i in range(0, len(slots_json), 1000):
    db["slots"].insert_many(slots_json[i:i+1000])
client.close()
print("Slots uploaded to MongoDB.")

# Read back from MongoDB
client = get_mongo_client(**mongodb_args)
df_slots_mongo = get_mongo_dataframe(
    client, mongodb_args["db_name"], "slots", {}
)
df_dim_slots = df_slots_mongo[
    ['slot_id','appointment_date','appointment_time','is_available']
].copy()
df_dim_slots = df_dim_slots.drop_duplicates().reset_index(drop=True)
df_dim_slots.insert(0, 'slot_key', df_dim_slots.index + 1)

set_dataframe(df_dim_slots, "workspace.default.dim_slots")

Slots uploaded to MongoDB.
Saved 'workspace.default.dim_slots': 10000 rows


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.dim_age_groups")

df_appts = pd.read_csv(f"{VOLUME_PATH}/appointments.csv")
df_dim_age = df_appts[['age_group']].drop_duplicates().reset_index(drop=True)
df_dim_age.insert(0, 'age_group_key', df_dim_age.index + 1)

set_dataframe(df_dim_age, "workspace.default.dim_age_groups")

Saved 'workspace.default.dim_age_groups': 16 rows


## Step 4: Prepare Streaming JSON Files
Segment the appointments fact data into 3 separate JSON files to simulate
a real-time streaming source. Each file represents one streaming interval.
This implements the AutoLoader pattern required for the Bronze layer.

In [0]:
# Clear and recreate streaming directory
dbutils.fs.rm(STREAMING_INPUT, recurse=True)
dbutils.fs.mkdirs(STREAMING_INPUT)

df_source = pd.read_csv(f"{VOLUME_PATH}/appointments.csv")
total = len(df_source)
third = total // 3

# Split into exactly 3 JSON interval files
intervals = [
    df_source.iloc[0:third],
    df_source.iloc[third:third*2],
    df_source.iloc[third*2:]
]

for idx, interval in enumerate(intervals):
    path = f"{VOLUME_PATH}/streaming_json/interval_{idx+1}.json"
    interval.to_json(path, orient='records', indent=2)
    print(f"interval_{idx+1}.json — {len(interval)} rows")

interval_1.json — 37162 rows
interval_2.json — 37162 rows
interval_3.json — 37164 rows


## Step 5: Bronze Layer — Raw Streaming Ingestion
Use Spark AutoLoader cloudFiles to ingest streaming JSON files
into the Bronze Delta Table. This represents raw, unprocessed fact data
exactly as it arrives from the source system.

In [0]:
# Drop existing bronze table and checkpoint
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_PATH}")
dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
dbutils.fs.rm(f"{VOLUME_PATH}/schema_hints", recurse=True)

# Define schema
bronze_schema = StructType([
    StructField("appointment_id",       LongType(),    True),
    StructField("slot_id",              LongType(),    True),
    StructField("scheduling_date",      StringType(),  True),
    StructField("appointment_date",     StringType(),  True),
    StructField("appointment_time",     StringType(),  True),
    StructField("scheduling_interval",  IntegerType(), True),
    StructField("status",               StringType(),  True),
    StructField("check_in_time",        StringType(),  True),
    StructField("appointment_duration", DoubleType(),  True),
    StructField("start_time",           StringType(),  True),
    StructField("end_time",             StringType(),  True),
    StructField("waiting_time",         DoubleType(),  True),
    StructField("patient_id",           LongType(),    True),
    StructField("sex",                  StringType(),  True),
    StructField("age",                  IntegerType(), True),
    StructField("age_group",            StringType(),  True),
])

# Read JSON files using standard readStream (not AutoLoader)
df_bronze_stream = (
    spark.readStream
        .format("json")
        .schema(bronze_schema)
        .option("maxFilesPerTrigger", 1)
        .load(STREAMING_INPUT)
)

# Add ingestion timestamp
df_bronze_stream = df_bronze_stream.withColumn(
    "ingestion_time", current_timestamp()
)

# Write to Bronze Delta Table
bronze_query = (
    df_bronze_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(BRONZE_PATH)
)

bronze_query.awaitTermination()

count = spark.sql(f"SELECT COUNT(*) as n FROM {BRONZE_PATH}").collect()[0]["n"]
print(f"Bronze layer complete: {count} rows ingested")
spark.sql(f"SELECT * FROM {BRONZE_PATH} LIMIT 5").show()

Bronze layer complete: 111488 rows ingested
+--------------+-------+---------------+----------------+----------------+-------------------+--------------+-------------+--------------------+----------+--------+------------+----------+------+---+---------+--------------------+
|appointment_id|slot_id|scheduling_date|appointment_date|appointment_time|scheduling_interval|        status|check_in_time|appointment_duration|start_time|end_time|waiting_time|patient_id|   sex|age|age_group|      ingestion_time|
+--------------+-------+---------------+----------------+----------------+-------------------+--------------+-------------+--------------------+----------+--------+------------+----------+------+---+---------+--------------------+
|         37126|  34528|     2018-04-15|      2018-04-24|        09:45:00|                  9|did not attend|         NULL|                NULL|      NULL|    NULL|        NULL|      8723|  Male| 82|    80-84|2026-05-09 00:36:...|
|         37066|  34529|     201

## Step 6: Silver Layer - Cleanse & Join with Dimensions
Transform raw Bronze data by:
1. Adding a `date_key` for joining with `dim_date`
2. Joining with all dimension tables to resolve surrogate keys
3. Selecting only the columns needed for the fact table

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {SILVER_PATH}")

# Load bronze data
df_bronze = spark.table(BRONZE_PATH)

# Add date_key
df_bronze = df_bronze.withColumn(
    "date_key",
    date_format(
        to_date(col("appointment_date"), "yyyy-MM-dd"), "yyyyMMdd"
    ).cast("int")
)

# Load dimension tables
df_patient_keys  = spark.table("workspace.default.dim_patients").select("patient_key", "patient_id")
df_slot_keys     = spark.table("workspace.default.dim_slots").select("slot_key", "slot_id")
df_age_keys      = spark.table("workspace.default.dim_age_groups").select("age_group_key", "age_group")
df_date_keys     = spark.table("workspace.default.dim_date").select("date_key")

# Join fact data with all dimensions (Silver = integrated layer)
df_silver = (df_bronze
    .join(df_patient_keys, on="patient_id", how="left")
    .join(df_slot_keys,    on="slot_id",    how="left")
    .join(df_age_keys,     on="age_group",  how="left")
    .select(
        "appointment_id",
        "patient_key",
        "slot_key",
        "date_key",
        "age_group_key",
        "scheduling_interval",
        "status",
        "appointment_duration",
        "waiting_time",
        "ingestion_time"
    )
)

# Save Silver Delta Table
save_delta_table(df_silver, SILVER_PATH)
print("Silver layer complete.")
spark.sql(f"SELECT * FROM {SILVER_PATH} LIMIT 5").show()

Saved 'workspace.default.silver_appointments': 111488 rows
+--------------+-----------+--------+--------+-------------+-------------------+--------------+--------------------+------------+--------------------+
|appointment_id|patient_key|slot_key|date_key|age_group_key|scheduling_interval|        status|appointment_duration|waiting_time|      ingestion_time|
+--------------+-----------+--------+--------+-------------+-------------------+--------------+--------------------+------------+--------------------+
|           138|       8285|       1|20150101|            1|                  4|did not attend|                NULL|        NULL|2026-05-09 00:36:...|
|           146|       5972|      23|20150101|            2|                  3|did not attend|                NULL|        NULL|2026-05-09 00:36:...|
|            21|       6472|      24|20150101|            3|                 15|      attended|                 5.2|         1.2|2026-05-09 00:36:...|
|           233|       5376|      2

## Step 7: Gold Layer - Final Fact Table
Promote the Silver table to the Gold layer - the final,
production-ready fact table optimized for analytical queries.
This is the core of the dimensional Data Lakehouse.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {GOLD_PATH}")

df_gold = spark.table(SILVER_PATH)
save_delta_table(df_gold, GOLD_PATH)

print("Gold layer complete.")
spark.sql(f"SELECT * FROM {GOLD_PATH} LIMIT 10").show()

Saved 'workspace.default.fact_appointments': 111488 rows
Gold layer complete.
+--------------+-----------+--------+--------+-------------+-------------------+--------------+--------------------+------------+--------------------+
|appointment_id|patient_key|slot_key|date_key|age_group_key|scheduling_interval|        status|appointment_duration|waiting_time|      ingestion_time|
+--------------+-----------+--------+--------+-------------+-------------------+--------------+--------------------+------------+--------------------+
|           138|       8285|       1|20150101|            1|                  4|did not attend|                NULL|        NULL|2026-05-09 00:36:...|
|           146|       5972|      23|20150101|            2|                  3|did not attend|                NULL|        NULL|2026-05-09 00:36:...|
|            21|       6472|      24|20150101|            3|                 15|      attended|                 5.2|         1.2|2026-05-09 00:36:...|
|           233|

## Step 8: Analytical Queries
Demonstrate the business value of the Data Lakehouse by running
analytical queries that summarize appointments across dimensions.
These queries illustrate the post hoc diagnostic analysis capability
of the dimensional model.

In [0]:
# Query 1: Appointments by status
print("Query 1: Appointments by Status")
spark.sql("""
    SELECT status,
           COUNT(*) AS total_appointments,
           ROUND(AVG(waiting_time), 2) AS avg_waiting_mins
    FROM workspace.default.fact_appointments
    GROUP BY status
    ORDER BY total_appointments DESC
""").show()

# Query 2: Appointments by age group
print("Query 2: Appointments by Age Group")
spark.sql("""
    SELECT ag.age_group,
           COUNT(*) AS total_appointments,
           ROUND(AVG(f.waiting_time), 2) AS avg_waiting_mins
    FROM workspace.default.fact_appointments f
    JOIN workspace.default.dim_age_groups ag
        ON f.age_group_key = ag.age_group_key
    GROUP BY ag.age_group
    ORDER BY total_appointments DESC
""").show()

# Query 3: Appointments by insurance type and year
print("Query 3: Appointments by Insurance Type and Year")
spark.sql("""
    SELECT p.insurance,
           d.calendar_year,
           COUNT(*) AS total_appointments,
           ROUND(AVG(f.waiting_time), 2) AS avg_waiting_mins
    FROM workspace.default.fact_appointments f
    JOIN workspace.default.dim_patients p
        ON f.patient_key = p.patient_key
    JOIN workspace.default.dim_date d
        ON f.date_key = d.date_key
    GROUP BY p.insurance, d.calendar_year
    ORDER BY d.calendar_year, total_appointments DESC
""").show()

# Query 4: Monthly appointment trends
print("Query 4: Monthly Appointment Trends")
spark.sql("""
    SELECT d.calendar_year,
           d.month_name,
           d.month_of_year,
           COUNT(*) AS total_appointments,
           ROUND(AVG(f.waiting_time), 2) AS avg_waiting_mins
    FROM workspace.default.fact_appointments f
    JOIN workspace.default.dim_date d
        ON f.date_key = d.date_key
    GROUP BY d.calendar_year, d.month_name, d.month_of_year
    ORDER BY d.calendar_year, d.month_of_year
""").show()

Query 1: Appointments by Status
+--------------+------------------+----------------+
|        status|total_appointments|avg_waiting_mins|
+--------------+------------------+----------------+
|      attended|             86032|           44.09|
|     cancelled|             18254|            NULL|
|did not attend|              6615|            NULL|
|       unknown|               446|            NULL|
|     scheduled|               141|            NULL|
+--------------+------------------+----------------+

Query 2: Appointments by Age Group
+---------+------------------+----------------+
|age_group|total_appointments|avg_waiting_mins|
+---------+------------------+----------------+
|    75-79|              9844|           44.49|
|    60-64|              9649|           43.52|
|    70-74|              9485|           43.38|
|    65-69|              9133|           42.99|
|    55-59|              8955|            44.4|
|    80-84|              8140|           43.43|
|    50-54|            

## Summary
1. **Multi-source data integration** -> Azure SQL, MongoDB Atlas, and DBFS CSV files
2. **Spark AutoLoader** -> real-time JSON streaming ingestion across 3 intervals
3. **Bronze/Silver/Gold architecture** -> raw ingestion → cleansed joins → production facts
4. **Dimensional Data Lakehouse** -> star schema with dim_date, dim_patients, dim_slots, dim_age_groups
5. **Analytical queries** -> business value demonstrated across 4 summary queries